<a href="https://colab.research.google.com/github/nehakeer/Automation-of-Attendance-System-to-Recognize-a-Human-Frontal-Face-With-and-Without-Masks/blob/main/agents/langchain_agent_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install -q langchain==0.1.16 langchain-openai==0.0.8 langchain-community==0.0.32

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
db-dtypes 1.5.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1

In [25]:
!pip install --upgrade numpy -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.32 requires numpy<2,>=1, but you have numpy 2.4.2 which is incompatible.
langchain 0.1.16 requires numpy<2,>=1, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
db-dtypes 1.5.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.


In [37]:
from langchain_openai import ChatOpenAI
import os
import pandas as pd

from langchain.agents import AgentType, initialize_agent,load_tools
from langchain.tools import Tool


In [38]:
# # local model
# llm = ChatOpenAI(
#     openai_api_base="http://localhost:1234/v1",
#     openai_api_key="lm_studio",
#     model="llama-3.2-1b-instruct",
#     temperature=0.9
# )

In [48]:
llm = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key="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",
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.5
)

os.environ["SERPAPI_API_KEY"] = "9e5fbdfc4db1f8150b622823fcb40cd9f1d2f9149b4279f6dd67ba5f759eea45"


In [49]:
!git clone https://github.com/deepanrajm/deep_learning.git

fatal: destination path 'deep_learning' already exists and is not an empty directory.


In [50]:
EXCEL_PATH = "/content/deep_learning/agents/expense.xlsx"

In [51]:
def analyze_expenses_excel(query: str) -> str:
    """
    Analyze salary and expenses from Excel using pandas.
    """
    df = pd.read_excel(EXCEL_PATH)

    salary = df.loc[df["Category"].str.lower() == "salary", "Monthly_Amount"].sum()

    expenses_df = df[df["Category"].str.lower() != "salary"]

    total_expenses = expenses_df["Monthly_Amount"].sum()
    savings = salary - total_expenses
    savings_percentage = (savings / salary) * 100 if salary > 0 else 0

    top_expenses = (
        expenses_df
        .sort_values(by="Monthly_Amount", ascending=False)
        .head(3)[["Category", "Monthly_Amount"]]
        .to_dict(orient="records")
    )

    summary = {
        "monthly_salary": salary,
        "total_monthly_expenses": total_expenses,
        "monthly_savings": savings,
        "savings_percentage": round(savings_percentage, 2),
        "top_3_expense_categories": top_expenses
    }

    return str(summary)


In [52]:
expense_tool = Tool(
    name="expense_analyzer",
    description=(
        "Analyze salary and monthly expenses from Excel and return "
        "savings, spending breakdown, and high expense categories."
    ),
    func=analyze_expenses_excel
)


In [53]:
pip install google-search-results

In [54]:
tools = load_tools(
    ["serpapi", "llm-math"],
    llm=llm
)

tools.append(expense_tool)


In [55]:
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)


In [56]:
query = """
This is my monthly salary and expense sheet.
Please analyze it and tell me:
1. How much I am saving
2. Where I am overspending
3. How I can reduce expenses
4. How to improve my savings
"""

result = agent.run(query)
print(result)




> Entering new AgentExecutor chain...
Final Answer: Please provide your monthly salary and expense details (e.g., in an Excel file or a list of figures) so I can analyze your savings, identify overspending, suggest expense‑reduction strategies, and advise on improving your savings.

> Finished chain.
Please provide your monthly salary and expense details (e.g., in an Excel file or a list of figures) so I can analyze your savings, identify overspending, suggest expense‑reduction strategies, and advise on improving your savings.
